# 05 - Comparative Analysis (Jaccard Index & Venn Diagrams)

**Purpose:**
- Compute exact overlap (intersection) between IF and LOF anomalies.
- Compute the Jaccard Index as a formal metric of model agreement (Table 2).
- Generate Venn diagrams (Figure 6) and comparison heatmaps (Figure 7) using the corrected canonical labels, fixing BUG-06 (where a 2-feature subset was accidentally plotted in the original draft).

**Inputs:**
- `data/processed/anomaly_labels.parquet`

**Outputs:**
- `outputs/tables/table2_jaccard_overlap.csv`
- `outputs/figures/fig6_venn.png`
- `outputs/figures/fig7_heatmap.png`\n

In [ ]:
# Cell 01: Mount Storage & Bootstrap Paths
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Paper1_Revision')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

for folder in ['data/raw', 'data/interim', 'data/processed', 
               'outputs/figures', 'outputs/tables', 'outputs/models', 
               'outputs/notebook_exports']:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
    
print(f"Base directory set to: {BASE_DIR}")\n

In [ ]:
# Cell 02: Imports, Global Seeds & Style
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import jaccard_score
import warnings

# For Venn diagrams. Note: user may need to pip install matplotlib-venn
try:
    from matplotlib_venn import venn2, venn3
except ImportError:
    print("Installing matplotlib-venn...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib-venn"])
    from matplotlib_venn import venn2, venn3

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'figure.dpi': 300,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.autolayout': True
})\n

In [ ]:
# Cell 03: Load Anomaly Labels
labels_path = BASE_DIR / 'data' / 'processed' / 'anomaly_labels.parquet'
df_labels = pd.read_parquet(labels_path)
print(f"Labels shape: {df_labels.shape}")

# Convert -1/1 to sets of anomalous indices for easy intersection logic
if_anomalies = set(df_labels.index[df_labels['IF_Label'] == -1])
lof_auto_anomalies = set(df_labels.index[df_labels['LOF_Auto_Label'] == -1])
lof_05_anomalies = set(df_labels.index[df_labels['LOF_05_Label'] == -1])

print(f"IF Anomalies: {len(if_anomalies)}")
print(f"LOF (0.05) Anomalies: {len(lof_05_anomalies)}")\n

In [ ]:
# Cell 04: Calculate Jaccard Index & Overlap
def get_jaccard_and_overlap(setA, setB):
    intersection = len(setA.intersection(setB))
    union = len(setA.union(setB))
    jaccard = intersection / union if union > 0 else 0
    return intersection, union, jaccard

overlap_05, union_05, jaccard_05 = get_jaccard_and_overlap(if_anomalies, lof_05_anomalies)
overlap_auto, union_auto, jaccard_auto = get_jaccard_and_overlap(if_anomalies, lof_auto_anomalies)
overlap_lof, union_lof, jaccard_lof = get_jaccard_and_overlap(lof_05_anomalies, lof_auto_anomalies)

# PARITY ASSERTION: Must perfectly match manuscript Table 2
assert overlap_05 == 545, f"Parity failure: Expected 545 overlap IF and LOF(0.05), got {overlap_05}"
assert round(jaccard_05, 3) == 0.058, f"Parity failure: Expected Jaccard 0.058, got {round(jaccard_05, 3)}"

table2_data = [
    {'Model Pair': 'IF vs LOF (0.05)', 'Intersection': overlap_05, 'Union': union_05, 'Jaccard Index': round(jaccard_05, 3)},
    {'Model Pair': 'IF vs LOF (Auto)', 'Intersection': overlap_auto, 'Union': union_auto, 'Jaccard Index': round(jaccard_auto, 3)},
    {'Model Pair': 'LOF (0.05) vs LOF (Auto)', 'Intersection': overlap_lof, 'Union': union_lof, 'Jaccard Index': round(jaccard_lof, 3)}
]

df_table2 = pd.DataFrame(table2_data)
display(df_table2)

# Export
table2_path = BASE_DIR / 'outputs' / 'tables' / 'table2_jaccard_overlap.csv'
df_table2.to_csv(table2_path, index=False)
print(f"Exported Table 2 to {table2_path}")\n

In [ ]:
# Cell 05: Visualizations - Venn Diagrams (Figure 6)
# Fix for BUG-06: The original draft used a 2-feature subset for one of the circles. 
# Here we strictly use the canonical 7-feature labels.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: IF vs LOF (0.05)
v1 = venn2(
    subsets=(
        len(if_anomalies - lof_05_anomalies), 
        len(lof_05_anomalies - if_anomalies), 
        len(if_anomalies.intersection(lof_05_anomalies))
    ),
    set_labels=('Isolation Forest\n(n=5000)', 'LOF (0.05)\n(n=5000)'),
    set_colors=('#1f77b4', '#ff7f0e'),
    alpha=0.7,
    ax=axes[0]
)
axes[0].set_title('Anomaly Overlap: IF vs LOF (0.05)', fontsize=14)

# Subplot 2: IF vs LOF (Auto) vs LOF (0.05)
v2 = venn3(
    subsets=(
        len(if_anomalies - lof_auto_anomalies - lof_05_anomalies), # 100
        len(lof_auto_anomalies - if_anomalies - lof_05_anomalies), # 010
        len(if_anomalies.intersection(lof_auto_anomalies) - lof_05_anomalies), # 110
        len(lof_05_anomalies - if_anomalies - lof_auto_anomalies), # 001
        len(if_anomalies.intersection(lof_05_anomalies) - lof_auto_anomalies), # 101
        len(lof_auto_anomalies.intersection(lof_05_anomalies) - if_anomalies), # 011
        len(if_anomalies.intersection(lof_auto_anomalies).intersection(lof_05_anomalies)) # 111
    ),
    set_labels=('IF', 'LOF (Auto)', 'LOF (0.05)'),
    ax=axes[1]
)
axes[1].set_title('Three-way Model Comparison', fontsize=14)

plt.tight_layout()

# Export
fig6_path = BASE_DIR / 'outputs' / 'figures' / 'fig6_venn.png'
plt.savefig(fig6_path, dpi=300, bbox_inches='tight')
print(f"Exported Figure 6 to {fig6_path}")
plt.show()\n

In [ ]:
# Cell 06: Visualizations - Overlap Heatmap (Figure 7)
# A more formal Jaccard matrix heatmap

models = ['IF', 'LOF (0.05)', 'LOF (Auto)']
jaccard_matrix = np.zeros((3, 3))

# Compute pair-wise Jaccard for matrix
sets = [if_anomalies, lof_05_anomalies, lof_auto_anomalies]
for i in range(3):
    for j in range(3):
        if i == j:
            jaccard_matrix[i, j] = 1.0
        else:
            _, _, j_idx = get_jaccard_and_overlap(sets[i], sets[j])
            jaccard_matrix[i, j] = j_idx

plt.figure(figsize=(8, 6))
sns.heatmap(
    jaccard_matrix, 
    annot=True, 
    fmt=".3f", 
    cmap="Blues", 
    xticklabels=models, 
    yticklabels=models,
    vmin=0, vmax=1
)
plt.title("Pairwise Jaccard Similarity Index", pad=20)

fig7_path = BASE_DIR / 'outputs' / 'figures' / 'fig7_heatmap.png'
plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
print(f"Exported Figure 7 to {fig7_path}")
plt.show()\n